In [1]:
from collections import deque
from pathlib import Path
import re

LOG_PATTERN = re.compile(
    r'^\[(?P<timestamp>[^\]]+)\]\s+(?P<ip>\S+)\s+"(?P<request>[^"]*)"\s+(?P<status>\d{3})'
)


def parse_line(line: str) -> tuple[tuple[str, str], str, str] | None:
    match = LOG_PATTERN.match(line)
    if match:
        key = (match.group("timestamp"), match.group("ip"))
        request = match.group("request")
        status = match.group("status")
        return key, request, status
    return None


def get_latest_logs(log_dir: str | Path, pattern: str = "access_*.log", target_count: int = 200) -> list[str]:
    directory = Path(log_dir)
    files = sorted(directory.glob(pattern), reverse=True)
    collected = []
    last_key = None

    for file_path in files:
        with open(file_path, "r", encoding="utf-8", errors="replace") as f:
            lines = list(deque(f))

        for line in reversed(lines):
            parsed = parse_line(line)
            if parsed is not None:
                key, request, status = parsed

                if status == "404":
                    continue

                if request == "GET /" and status == "200":
                    continue

                if key == last_key:
                    continue
                last_key = key
            else:
                last_key = None

            collected.append(line)
            if len(collected) >= target_count:
                return collected

    return collected


if __name__ == "__main__":
    target_dir = "./"
    latest_logs = get_latest_logs(target_dir, "access_*.log", 500)

    for line in latest_logs:
        print(line, end="")

[2026/09/18 23:32:04] 2400:2200:948:6af2:d590:90db:1301:1b13 "GET /shift/20261001" 200 "Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/26.6 Mobile/15E148 Safari/604.1"
[2026/09/18 23:32:03] 2400:2200:948:6af2:d590:90db:1301:1b13 "GET /shift/20260930" 200 "Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/26.6 Mobile/15E148 Safari/604.1"
[2026/09/18 23:32:01] 2400:2200:948:6af2:d590:90db:1301:1b13 "GET /shift/20260929" 200 "Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/26.6 Mobile/15E148 Safari/604.1"
[2026/09/18 23:31:58] 2400:2200:948:6af2:d590:90db:1301:1b13 "GET /shift/20260928" 200 "Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/26.6 Mobile/15E148 Safari/604.1"
[2026/09/18 23:31:55] 2400:2200:948:6af2:d590:90db:1301:1b13 "GET /shift/20260927" 200 "Mozilla/5.0 